# Data Preparation Notebook
## Predicting Event Attendance Using Logistic Regression
In this notebook, I will merge the data into a single dataset ready for modelling.

### What I'm predicting
I want to predict whether a user will mark themselves as **interested** in an event (1 = yes, 0 = no) (binary classification).

### Where the data comes from
The data is from a Kaggle competition based on a real social event platform. It has been split across multiple files.

| File | What it contains |
|------|------------------|
| `train.csv` | The core data: which user saw which event, and whether they were interested |
| `events_matched.csv` | Details about each event (time, location) - filtered down to the events that appear in train.csv |
| `users.csv` | Details about each user (age, gender, location) |
| `event_attendees_sample.csv` | How many people said yes/no/maybe to each event |

## Step 1: Import Libraries
- **pandas**: for loading and manipulating data tables
- **numpy**: for numerical operations (e.g. filling missing values with the median)

In [1]:
import pandas as pd
import numpy as np
import zipfile

print('Libraries loaded successfully')

Libraries loaded successfully


## Step 2: Load the Raw Data
This is the core file, each row is one (user, event) pair

In [2]:
train = pd.read_csv('train.csv') # Loads train.csv file into a pandas table

print('=== TRAIN.CSV ===')
print(f'Shape: {train.shape[0]} rows, {train.shape[1]} columns')
print(train.head())
print()
print('Target variable distribution:')
print(train['interested'].value_counts())
print(f'=> {100*train["interested"].mean():.1f}% of rows are "interested = 1"')

=== TRAIN.CSV ===
Shape: 15398 rows, 6 columns
      user       event  invited                         timestamp  interested  \
0  3044012  1918771225        0  2012-10-02 15:53:05.754000+00:00           0   
1  3044012  1502284248        0  2012-10-02 15:53:05.754000+00:00           0   
2  3044012  2529072432        0  2012-10-02 15:53:05.754000+00:00           1   
3  3044012  3072478280        0  2012-10-02 15:53:05.754000+00:00           0   
4  3044012  1390707377        0  2012-10-02 15:53:05.754000+00:00           0   

   not_interested  
0               0  
1               0  
2               0  
3               0  
4               0  

Target variable distribution:
interested
0    11267
1     4131
Name: count, dtype: int64
=> 26.8% of rows are "interested = 1"


**Explaining train.csv:**
- Each row = one user encountering one event
- `invited`: was the user personally invited? (1 = yes, 0 = no)
- `timestamp`: when the user saw the event listing
- `interested`: the **target variable**
- `not_interested`: the opposite signal — this will not be used (avoid data leakage)

There is a **class imbalance** as only ~27% of rows are "interested". This means the model will be better at predicting 0 than 1.

In [3]:
# Load event details
events = pd.read_csv('events_matched.csv')

print('=== EVENTS_MATCHED.CSV ===')
print(f'Shape: {events.shape[0]} rows, {events.shape[1]} columns')
print()
# We only show the most relevant columns (there are 110 total)
print('Key columns:')
print(events[['event_id', 'start_time', 'lat', 'lng', 'city', 'country']].head())
print()
print(f'lat/lng missing: {events["lat"].isnull().sum()} rows ({100*events["lat"].isnull().sum()/len(events):.1f}%)')

=== EVENTS_MATCHED.CSV ===
Shape: 8846 rows, 110 columns

Key columns:
     event_id                start_time  lat  lng city country
0  3928440935  2012-11-05T00:00:00.001Z  NaN  NaN  NaN     NaN
1  2582345152  2012-10-30T00:00:00.001Z  NaN  NaN  NaN     NaN
2  1212611096  2012-11-16T00:00:00.001Z  NaN  NaN  NaN     NaN
3  3689283674  2012-11-02T20:00:00.003Z  NaN  NaN  NaN     NaN
4  2584113432  2012-10-31T00:00:00.001Z  NaN  NaN  NaN     NaN

lat/lng missing: 3556 rows (40.2%)


**Explaining events:**
- `start_time`: when the event happens — will be used to extract the hour and day of week
- `lat` / `lng`: the geographic coordinates of the event. As **36% are missing** in the original dataset, `timezone` will be used as a location estimate instead
- `c_1` to `c_100`: this is likely word counts from event descriptions, as the actual data is hidden, this will be ignored

In [5]:
# Load user demographics
users = pd.read_csv('users.csv')

print('=== USERS.CSV ===')
print(f'Shape: {users.shape[0]} rows, {users.shape[1]} columns')
print(users.head())
print()
print('Missing values:')
print(users.isnull().sum())

=== USERS.CSV ===
Shape: 38209 rows, 7 columns
      user_id locale birthyear  gender                  joinedAt  \
0  3197468391  id_ID      1993    male  2012-10-02T06:40:55.524Z   
1  3537982273  id_ID      1992    male  2012-09-29T18:03:12.111Z   
2   823183725  en_US      1975    male  2012-10-06T03:14:07.149Z   
3  1872223848  en_US      1991  female  2012-11-04T08:59:43.783Z   
4  3429017717  id_ID      1995  female  2012-09-10T16:06:53.132Z   

             location  timezone  
0    Medan  Indonesia     480.0  
1    Medan  Indonesia     420.0  
2  Stratford  Ontario    -240.0  
3        Tehran  Iran     210.0  
4                 NaN     420.0  

Missing values:
user_id         0
locale          0
birthyear    1492
gender        109
joinedAt       58
location     5465
timezone      436
dtype: int64


In [6]:
# Load event attendees — how many people responded to each event
attendees = pd.read_csv('event_attendees_sample.csv')

print('=== EVENT_ATTENDEES_SAMPLE.CSV ===')
print(f'Shape: {attendees.shape[0]} rows, {attendees.shape[1]} columns')
print(attendees.head())
print()

=== EVENT_ATTENDEES_SAMPLE.CSV ===
Shape: 5000 rows, 5 columns
        event                                                yes  \
0  1159822043  1975964455 252302513 4226086795 3805886383 142...   
1   686467261  2394228942 2686116898 1056558062 3792942231 41...   
2  1186208412                                                NaN   
3  2621578336                                                NaN   
4   855842686  2406118796 3550897984 294255260 1125817077 109...   

                                               maybe  \
0  2733420590 517546982 1350834692 532087573 5831...   
1  1498184352 645689144 3770076778 331335845 4239...   
2                              3320380166 3810793697   
3                                                NaN   
4  2671721559 1761448345 2356975806 2666669465 10...   

                                             invited                     no  
0  1723091036 3795873583 4109144917 3560622906 31...  3575574655 1077296663  
1  1788073374 733302094 1830571649 

- Each row = one event
- Each column contains a space-separated list of user IDs
- The number of *yes* responses will be counted as the event popularity measure.

## Step 3: Data Transformation

### 3a: Event Popularity (from attendees file)

Each row in the attendees file has a space-separated list of user IDs in the `yes` column. Count how many IDs are in that list to get a **popularity score** for each event.

In [7]:
# Count the number of 'yes' responses per event
# The yes column looks like: '123456 789012 345678' — space-separated user IDs
# We split on spaces and count how many IDs there are
attendees['yes_count'] = attendees['yes'].fillna('').apply(
    lambda x: len(x.split()) if x else 0
)

event_popularity = attendees[['event', 'yes_count']].rename(columns={'event': 'event_id'})

print('Event popularity (yes_count) distribution:')
print(event_popularity['yes_count'].describe())
print()
print('Sample:')
print(event_popularity.sort_values('yes_count', ascending=False).head(10))

Event popularity (yes_count) distribution:
count    5000.000000
mean       28.297800
std        92.840673
min         0.000000
25%         7.000000
50%        15.000000
75%        30.000000
max      5709.000000
Name: yes_count, dtype: float64

Sample:
        event_id  yes_count
4025    90124184       5709
1341  2585327580       1048
438   2938463739        863
4635  2978312627        766
4055  1554701042        747
263    416330287        633
2722  3833592318        532
218    235476646        459
3612  2610561518        456
4691  4168094163        433


### 3b: Event Time Features (from events file)

Extract two time features from the event start time:
- **event_hour**: what hour of the day does the event start? (0-23)
- **event_dow**: what day of the week? (0=Monday, 6=Sunday)

These capture whether events at certain times (e.g. weekend evenings) attract more interest.

In [8]:
# Parse the start_time column as a proper datetime object
# format='mixed' handles slight variations in the timestamp format
events['start_time'] = pd.to_datetime(events['start_time'], format='mixed', utc=True)

# Extract hour and day of week
events['event_hour'] = events['start_time'].dt.hour
events['event_dow']  = events['start_time'].dt.dayofweek  # 0=Monday, 6=Sunday

# Keep only what we need and merge in popularity
events_features = events[['event_id', 'event_hour', 'event_dow']].copy()
events_features = events_features.merge(event_popularity, on='event_id', how='left')
events_features['yes_count'] = events_features['yes_count'].fillna(0)

print('Event features created:')
print(events_features.head(10))
print()
print('event_hour distribution (0=midnight, 12=noon):')
print(events_features['event_hour'].value_counts().sort_index())

Event features created:
     event_id  event_hour  event_dow  yes_count
0  3928440935           0          0        0.0
1  2582345152           0          1        1.0
2  1212611096           0          4        0.0
3  3689283674          20          4       26.0
4  2584113432           0          2        0.0
5  3365728297           0          2        0.0
6  1609864127          22          1        0.0
7  2608543989          17          0       44.0
8   298169907           0          5        0.0
9  1922719636           0          2        2.0

event_hour distribution (0=midnight, 12=noon):
event_hour
0     2148
1      558
2      964
3      673
4      463
5      382
6      218
7      154
8      191
9      137
10     120
11     185
12     295
13     251
14     189
15     173
16     170
17     228
18     222
19     288
20     198
21     177
22     214
23     248
Name: count, dtype: int64


### 3c: User Features (from users file)

Extract four user-level features:
- **age**: calculated from birth year (data is from 2012)
- **is_male**: binary flag (1 = male, 0 = female/other)
- **is_en_US**: binary flag for US English locale - a rough proxy for being based in the US
- **timezone**: the user's UTC timezone offset in minutes — used as a location proxy

Missing values in age and timezone are filled with the **median** (middle value of all users).

In [9]:
users_clean = users[['user_id', 'birthyear', 'gender', 'locale', 'timezone']].copy()

# birthyear is stored as a string — convert to number, invalid entries become NaN
users_clean['birthyear'] = pd.to_numeric(users_clean['birthyear'], errors='coerce')

# Calculate age (data is from 2012)
users_clean['age'] = 2012 - users_clean['birthyear']

# Fill the 1% of missing ages with the median age
median_age = users_clean['age'].median()
users_clean['age'] = users_clean['age'].fillna(median_age)

# Create binary features from categorical columns
users_clean['is_male']  = (users_clean['gender'] == 'male').astype(int)
users_clean['is_en_US'] = (users_clean['locale'] == 'en_US').astype(int)

# Fill missing timezone values with median
median_tz = users_clean['timezone'].median()
users_clean['timezone'] = users_clean['timezone'].fillna(median_tz)

print('User features created:')
print(users_clean[['user_id', 'age', 'is_male', 'is_en_US', 'timezone']].head(10))
print()
print(f'Median age used for imputation: {median_age:.1f} years')
print(f'Proportion male: {users_clean["is_male"].mean():.1%}')
print(f'Proportion en_US: {users_clean["is_en_US"].mean():.1%}')

User features created:
      user_id   age  is_male  is_en_US  timezone
0  3197468391  19.0        1         0     480.0
1  3537982273  20.0        1         0     420.0
2   823183725  37.0        1         1    -240.0
3  1872223848  21.0        0         1     210.0
4  3429017717  17.0        0         0     420.0
5   627175141  39.0        0         0     240.0
6  2752000443  18.0        1         0     420.0
7  3473687777  47.0        0         0     420.0
8  2966052962  33.0        1         0     420.0
9   264876277  24.0        0         0     420.0

Median age used for imputation: 21.0 years
Proportion male: 60.8%
Proportion en_US: 44.7%


### 3d: Train Features (from train file)

Extract two additional features directly from the training data:
- **invite_hour**: what hour of the day was the event recommendation shown to the user?
- **event_train_count**: how many times does this event appear in the training data? This is the second measure of event popularity - more appearances suggests the event was shown to more users

In [10]:
# Parse timestamp
train['timestamp'] = pd.to_datetime(train['timestamp'], format='mixed', utc=True)
train['invite_hour'] = train['timestamp'].dt.hour

# Count how many times each event appears in the training data
event_train_count = train.groupby('event').size().reset_index(name='event_train_count')

print('invite_hour distribution:')
print(train['invite_hour'].value_counts().sort_index())
print()
print('event_train_count (top 10 most common events):')
print(event_train_count.sort_values('event_train_count', ascending=False).head(10))

invite_hour distribution:
invite_hour
0     567
1     567
2     716
3     718
4     751
5     959
6     816
7     759
8     791
9     706
10    628
11    567
12    753
13    772
14    670
15    714
16    651
17    506
18    501
19    445
20    459
21    463
22    463
23    456
Name: count, dtype: int64

event_train_count (top 10 most common events):
           event  event_train_count
1944   955398943                242
4190  2007279414                196
5286  2529072432                187
2576  1269035551                147
3283  1600413013                114
2839  1390707377                 99
2186  1076364848                 98
3139  1532377761                 96
4486  2149464820                 89
555    268233790                 80


## Step 4: Merge Everything Together

Join all four sources into a single dataset.

In [11]:
# Start with train, merge event features
df = train.merge(events_features, left_on='event', right_on='event_id', how='left')

# Merge user features
df = df.merge(
    users_clean[['user_id', 'age', 'is_male', 'is_en_US', 'timezone']],
    left_on='user', right_on='user_id', how='left'
)

# Merge event train count
df = df.merge(event_train_count, on='event', how='left')

print('Merged dataset shape:', df.shape)
print()
print('All columns available:')
print(df.columns.tolist())

Merged dataset shape: (15398, 17)

All columns available:
['user', 'event', 'invited', 'timestamp', 'interested', 'not_interested', 'invite_hour', 'event_id', 'event_hour', 'event_dow', 'yes_count', 'user_id', 'age', 'is_male', 'is_en_US', 'timezone', 'event_train_count']


## Step 5: Select Final Features and Check Quality

Now select our final 10 features and verify that there are no missing values. These are the features the logistic regression model will train on.

| Feature | Source | What it captures |
|---------|--------|------------------|
| `invited` | train.csv | Was the user personally invited? |
| `event_hour` | events | What hour does the event start? |
| `event_dow` | events | What day of the week? (0=Mon, 6=Sun) |
| `yes_count` | attendees | How many people said yes to this event? |
| `event_train_count` | train.csv | How widely was this event shown? |
| `age` | users | User age in 2012 |
| `is_male` | users | Is the user male? (1/0) |
| `is_en_US` | users | Is the user in US English locale? (1/0) |
| `timezone` | users | User timezone offset (location proxy) |
| `invite_hour` | train.csv | What hour was the listing shown to the user? |

In [12]:
feature_cols = [
    'invited',           # Was the user personally invited?
    'event_hour',        # Hour the event starts (0-23)
    'event_dow',         # Day of week (0=Mon, 6=Sun)
    'yes_count',         # Number of yes RSVPs from attendees file
    'event_train_count', # How many users were shown this event
    'age',               # User age
    'is_male',           # Gender binary flag
    'is_en_US',          # Locale binary flag
    'timezone',          # User timezone (location proxy)
    'invite_hour',       # Hour the listing was shown to the user
]

target_col = 'interested'

# Create final clean dataset
final_df = df[feature_cols + [target_col, 'user', 'event']].copy()

print('=== FINAL DATASET QUALITY CHECK ===')
print(f'Total rows: {len(final_df)}')
print(f'Total features: {len(feature_cols)}')
print()
print('Missing values per feature:')
print(final_df[feature_cols].isnull().sum())
print()
print('Feature summary statistics:')
print(final_df[feature_cols].describe().round(2))

=== FINAL DATASET QUALITY CHECK ===
Total rows: 15398
Total features: 10

Missing values per feature:
invited              0
event_hour           0
event_dow            0
yes_count            0
event_train_count    0
age                  0
is_male              0
is_en_US             0
timezone             0
invite_hour          0
dtype: int64

Feature summary statistics:
        invited  event_hour  event_dow  yes_count  event_train_count  \
count  15398.00    15398.00   15398.00   15398.00           15398.00   
mean       0.04        6.93       3.85       9.42              20.00   
std        0.20        6.93       1.76      37.71              46.82   
min        0.00        0.00       0.00       0.00               1.00   
25%        0.00        1.00       3.00       0.00               1.00   
50%        0.00        4.00       4.00       0.00               2.00   
75%        0.00       13.00       5.00       0.00              10.00   
max        1.00       23.00       6.00    1048.00 

In [13]:
# Final check: target variable
print('=== TARGET VARIABLE ===')
counts = final_df[target_col].value_counts()
print(counts)
print()
print(f'Class balance: {100*final_df[target_col].mean():.1f}% interested, {100*(1-final_df[target_col].mean()):.1f}% not interested')

=== TARGET VARIABLE ===
interested
0    11267
1     4131
Name: count, dtype: int64

Class balance: 26.8% interested, 73.2% not interested

NOTE: This class imbalance (roughly 1:3) means the model will naturally
be better at predicting "not interested" than "interested".
We should pay close attention to recall for class 1 in our evaluation.


- This class imbalance (roughly 1:3) means the model will naturally be better at predicting "not interested" than "interested".
- This means it is important to pay close attention to the recall value when the model is tested.

## Step 6: Save the Dataset

Save the final clean dataset as a CSV.

In [14]:
# Save to CSV — this is our training dataset for the repository
final_df.to_csv('event_attendance_dataset.csv', index=False)

print('Dataset saved as event_attendance_dataset.csv')
print(f'File contains {len(final_df)} rows and {len(final_df.columns)} columns')
print()
print('Columns saved:')
for col in final_df.columns:
    print(f'  - {col}')
print()
print('This file is ready for modelling in the next notebook.')

Dataset saved as event_attendance_dataset.csv
File contains 15398 rows and 13 columns

Columns saved:
  - invited
  - event_hour
  - event_dow
  - yes_count
  - event_train_count
  - age
  - is_male
  - is_en_US
  - timezone
  - invite_hour
  - interested
  - user
  - event

This file is ready for modelling in the next notebook.


## Summary

A clean dataset has been built by:

1. **Loading** four raw data files from the Kaggle event recommendation dataset
2. **Engineering features** from each: time components from timestamps, popularity counts from attendee lists, and demographic flags from user profiles
3. **Handling missing data**: lat/lng were too sparse (36% missing) and dropped in favour of timezone as a location proxy; small gaps in age and timezone were filled with medians
4. **Merging** everything into a single 15,398-row dataset with 10 features and zero missing values
5. **Noting the class imbalance**: only 26.8% of rows are 'interested = 1', which we will need to consider during model evaluation

The dataset is saved as `event_attendance_dataset.csv` and is ready for logistic regression modelling.